In [ ]:
import pandas as pd
import numpy as np
import hashlib, json, os
from datetime import datetime

os.makedirs("data", exist_ok=True)

print("=" * 60)
print("  FEATURE ENGINEERING — APPLICATION DES RECOMMANDATIONS EDA")
print("=" * 60)

# ─────────────────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────────────────
print("\n[1/4] Chargement du dataset original...")
df = pd.read_csv("data/transactions_bancaires.csv")
print(f"   {len(df):,} transactions | {df.shape[1]} colonnes")
print(f"   Fraudes : {df['fraude'].sum():,} ({df['fraude'].mean()*100:.2f}%)")


In [ ]:
# ─────────────────────────────────────────────────────────
# 2. CORRECTIONS
# ─────────────────────────────────────────────────────────
print("\n[2/4] Corrections des artefacts...")

# Correction 1 : valeurs négatives de delta_min_last_tx
n_neg = (df['delta_min_last_tx'] < 0).sum()
df['delta_min_last_tx'] = df['delta_min_last_tx'].abs()
print(f"   [FIX] delta_min_last_tx : {n_neg:,} valeurs négatives → abs()")
assert (df['delta_min_last_tx'] < 0).sum() == 0, "Correction échouée"
print(f"         Min après correction : {df['delta_min_last_tx'].min():.1f}  "
      f"Max : {df['delta_min_last_tx'].max():.0f}")


In [ ]:
# ─────────────────────────────────────────────────────────
# 3. TRANSFORMATIONS LOGARITHMIQUES
# ─────────────────────────────────────────────────────────
print("\n[3/4] Transformations et nouvelles features...")

# Log-transform pour réduire l'asymétrie
df['log_montant_mad'] = np.log1p(df['montant_mad'])
df['log_delta_km']    = np.log1p(df['delta_km'])

print(f"   [NEW] log_montant_mad   : skewness {df['montant_mad'].skew():.2f} → {df['log_montant_mad'].skew():.2f}")
print(f"   [NEW] log_delta_km      : skewness {df['delta_km'].skew():.2f} → {df['log_delta_km'].skew():.2f}")

# Feature binaire nuit
df['est_nuit'] = (df['heure'] <= 5).astype(int)
nuit_fraud_rate = df[df['est_nuit']==1]['fraude'].mean() * 100
nuit_day_rate   = df[df['est_nuit']==0]['fraude'].mean() * 100
print(f"   [NEW] est_nuit          : taux fraude nuit={nuit_fraud_rate:.1f}%  jour={nuit_day_rate:.1f}%")

# Vitesse physique — signal très fort
# Si delta_km / delta_min * 60 > 900 km/h → impossible en avion civil
df['vitesse_km_min'] = df['delta_km'] / (df['delta_min_last_tx'] + 1)
v_fraud = df[df['fraude']==1]['vitesse_km_min'].mean()
v_legit = df[df['fraude']==0]['vitesse_km_min'].mean()
print(f"   [NEW] vitesse_km_min    : moy fraude={v_fraud:.3f}  moy legit={v_legit:.3f}  ratio={v_fraud/v_legit:.1f}×")

# Interaction montant × étranger
df['montant_x_etranger'] = df['montant_mad'] * df['est_etranger']
mx_fraud = df[df['fraude']==1]['montant_x_etranger'].mean()
mx_legit = df[df['fraude']==0]['montant_x_etranger'].mean()
print(f"   [NEW] montant_x_etranger: moy fraude={mx_fraud:.0f}  moy legit={mx_legit:.0f}")

# Interaction risque horaire × fréquence transactions
df['risque_x_nb_tx'] = df['risque_horaire'] * df['nb_tx_1h']
rx_fraud = df[df['fraude']==1]['risque_x_nb_tx'].mean()
rx_legit = df[df['fraude']==0]['risque_x_nb_tx'].mean()
print(f"   [NEW] risque_x_nb_tx   : moy fraude={rx_fraud:.3f}  moy legit={rx_legit:.3f}")

# ─────────────────────────────────────────────────────────
# 4. VÉRIFICATION ET SAUVEGARDE
# ─────────────────────────────────────────────────────────
print("\n[4/4] Vérification et sauvegarde...")

NEW_FEATURES = [
    'log_montant_mad', 'log_delta_km',
    'est_nuit', 'vitesse_km_min',
    'montant_x_etranger', 'risque_x_nb_tx'
]

# Vérifier pas de NaN introduits
for feat in NEW_FEATURES:
    n_nan = df[feat].isnull().sum()
    n_inf = np.isinf(df[feat]).sum()
    assert n_nan == 0, f"NaN dans {feat} !"
    assert n_inf == 0, f"Inf dans {feat} !"
print(f"   Aucun NaN/Inf dans les {len(NEW_FEATURES)} nouvelles features.")

# Corrélation des nouvelles features avec fraude
corr_new = df[NEW_FEATURES + ['fraude']].corr()['fraude'].drop('fraude')
print("\n   Corrélation des nouvelles features avec fraude :")
for feat, val in corr_new.sort_values(key=abs, ascending=False).items():
    bar = '█' * int(abs(val) * 30)
    print(f"     {feat:<25} {val:+.4f}  {bar}")

# Sauvegarde
output_path = "data/transactions_engineered.csv"
df.to_csv(output_path, index=False)
sha256 = hashlib.sha256(open(output_path, "rb").read()).hexdigest()

# Metadata pour blockchain
metadata = {
    "source_dataset":    "transactions_bancaires.csv",
    "engineered_dataset": output_path,
    "version":           "2.0.0",
    "n_rows":            len(df),
    "n_cols_original":   25,
    "n_cols_final":      len(df.columns),
    "new_features":      NEW_FEATURES,
    "fixes_applied":     ["abs(delta_min_last_tx)"],
    "transformations":   ["log1p(montant_mad)", "log1p(delta_km)"],
    "interactions":      ["montant_x_etranger", "risque_x_nb_tx"],
    "sha256":            sha256,
    "generated_at":      datetime.now().isoformat(),
    "feature_correlations_with_fraud": {
        k: round(float(v), 4)
        for k, v in corr_new.sort_values(key=abs, ascending=False).items()
    }
}
with open("data/engineered_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n   Colonnes : {25} → {len(df.columns)} (+{len(df.columns)-25} features)")
print(f"   Fichier  : {output_path}")
print(f"   SHA-256  : {sha256[:40]}...")

print("\n" + "=" * 60)
print("  FEATURE ENGINEERING TERMINÉ")
print("=" * 60)
print("  Prochaine étape : python3 02_train_models.py")
print("  (utilise automatiquement transactions_engineered.csv)")
